In [0]:
secret_scope_name = "fineguard-scope"

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

api_url = ctx.apiUrl().getOrElse(None)      # e.g. https://adb-...azuredatabricks.net
api_token = ctx.apiToken().getOrElse(None)  # personal access token for this session



import requests
import json

# ----------------------------------------
# Configuration
# ----------------------------------------
DATABRICKS_INSTANCE = api_url
DATABRICKS_TOKEN = api_token

scope_name = secret_scope_name
backend_type = "DATABRICKS"

kafka_bootstrap_servers = "pkc-75m1o.europe-west3.gcp.confluent.cloud:9092"
kafka_api_keys = "ZTHKMRDDBVXSCUKI"
kafka_api_secret = "cfltKBH+sh0vvHhsAGuj/BGX4kU0YmgGTTEyKKttHDx5PZubkIiqCy0OREO826Hg"
kafka_topic = "credit_card_transactions"

kafka_connection_details = json.dumps({
    "bootstrap_servers": kafka_bootstrap_servers,
    "api_keys": kafka_api_keys,
    "api_secret": kafka_api_secret,
    "topic": kafka_topic
})

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

# ----------------------------------------
# 1. Create secret scope (guarded)
# ----------------------------------------
create_scope_url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/create"
payload = {"scope": scope_name}

response = requests.post(create_scope_url, headers=headers, data=json.dumps(payload))

if response.status_code == 200:
    print(f"Secret scope '{scope_name}' created successfully.")
elif response.status_code == 400 and "RESOURCE_ALREADY_EXISTS" in response.text:
    print(f"Secret scope '{scope_name}' already exists — continuing.")
else:
    raise RuntimeError(f"Failed to create secret scope. Status: {response.status_code}, Response: {response.text}")

# ----------------------------------------
# 2. Store kafka_connection_details secret
# ----------------------------------------
put_url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/put"

kafka_put_payload = {
    "scope": scope_name,
    "key": "kafka_connection_details",
    "string_value": kafka_connection_details
}

kafka_put_response = requests.post(put_url, headers=headers, data=json.dumps(kafka_put_payload))

if kafka_put_response.status_code == 200:
    print("Secret 'kafka_connection_details' stored successfully.")
else:
    raise RuntimeError(f"Failed to store kafka_connection_details. Status: {kafka_put_response.status_code}, Response: {kafka_put_response.text}")

# ----------------------------------------
# 3. Verify retrieval
# ----------------------------------------
try:
    retrieved_json = dbutils.secrets.get(
        scope=secret_scope_name,
        key='kafka_connection_details'
    )

    print("Secret retrieved successfully.")

    parsed = json.loads(retrieved_json)
    print("Parsed JSON keys:", list(parsed.keys()))  # avoid printing values directly

except Exception as e:
    print("Secret verification failed:")
    print(str(e))


# COMMAND ----------

# ----------------------------------------
# 4. Store Gmail app password secret
# ----------------------------------------
scope = secret_scope_name
secret_key = 'gmail_api_key'
secret_value = 'pwun ygtd ibpi wsen'  # rotate this — see note below

gmail_put_payload = {
    "scope": scope,
    "key": secret_key,
    "string_value": secret_value
}

gmail_put_response = requests.post(put_url, headers=headers, data=json.dumps(gmail_put_payload))
print(dbutils.secrets.list("fineguard-scope"))
if gmail_put_response.status_code == 200:
    print(f"Secret '{secret_key}' created successfully in scope '{scope}'.")
else:
    print("Failed to create secret.")
    print("Status:", gmail_put_response.status_code)
    print("Response:", gmail_put_response.text)
    